# Olist Review Semantic Search (RAG)

**Author:** Pariya Lakani  
**Date:** 2026-11

An extension of the Review Insight Agent. The agent in `04_review_agent_organized.ipynb` 
can filter reviews by category, but it can only return **random** samples from each category.

This notebook adds **semantic search** so the agent can find reviews *about a specific theme* 
(packaging, communication, refund, etc.) across all 8,760 Portuguese reviews — regardless of category.

**Method:**
- Multilingual sentence-transformer embeddings (local, free, no API key)
- Chroma vector database for storage and search
- Cached to disk so re-runs are instant

**Contents**
1. Setup
2. Load data
3. Build embeddings (once)
4. Build the Chroma index
5. Search function + test
6. Example queries
7. Integration with the agent (next step)


## 1. Setup

Install the two packages we need: `sentence-transformers` (for multilingual embeddings) 
and `chromadb` (lightweight local vector DB).

In [1]:
%pip install -q sentence-transformers chromadb tqdm


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer
import chromadb

print("Imports OK")


Imports OK


## 2. Load the classified reviews

We use the same `sample_classified_full.csv` as the main notebook — 8,760 negative reviews with comments and their LLM-assigned category.

In [3]:
sample = pd.read_csv("../data/sample_classified_full.csv")
sample["category"] = sample["category"].replace({"seller_issue": "customer_service_issue"})

# Keep only what we need for RAG
reviews = sample[["order_id", "review_comment_message", "category", "is_late"]].copy()
reviews = reviews.dropna(subset=["review_comment_message"]).reset_index(drop=True)

print(f"Total reviews: {len(reviews)}")
print(f"Categories:\n{reviews['category'].value_counts()}")


Total reviews: 8760
Categories:
category
never_arrived             2284
wrong_item                1720
other                     1000
late_delivery              983
misleading_desc            791
broken_damaged             737
poor_quality               724
customer_service_issue     521
Name: count, dtype: int64


## 3. Build multilingual embeddings

The model `paraphrase-multilingual-MiniLM-L12-v2` is small (~400 MB) and handles 50+ languages including Portuguese. The first run downloads the model and encodes all 8,760 reviews — takes ~2–5 minutes on CPU. Results are cached to disk so subsequent runs skip encoding entirely.

In [4]:
EMBEDDINGS_PATH = Path("../data/review_embeddings.parquet")

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print(f"Model loaded. Embedding dimension: {model.get_sentence_embedding_dimension()}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded. Embedding dimension: 384


C:\Users\pariy\AppData\Local\Temp\ipykernel_27324\2241409115.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded. Embedding dimension: {model.get_sentence_embedding_dimension()}")


In [5]:
if EMBEDDINGS_PATH.exists():
    print(f"Loading cached embeddings from {EMBEDDINGS_PATH}...")
    cached = pd.read_parquet(EMBEDDINGS_PATH)
    embeddings = cached["embedding"].tolist()
    print(f"Loaded {len(embeddings)} embeddings")
else:
    print("Encoding all reviews (this takes 2-5 min on CPU)...")
    embeddings = model.encode(
        reviews["review_comment_message"].tolist(),
        show_progress_bar=True,
        batch_size=64,
    ).tolist()

    # Cache to parquet for next time
    cache_df = pd.DataFrame({
        "order_id": reviews["order_id"],
        "embedding": embeddings,
    })
    cache_df.to_parquet(EMBEDDINGS_PATH)
    print(f"Saved embeddings to {EMBEDDINGS_PATH}")


Loading cached embeddings from ..\data\review_embeddings.parquet...
Loaded 8760 embeddings


## 4. Build the Chroma index

Chroma is a lightweight local vector database. We create a persistent collection so the index survives between sessions.

In [6]:
CHROMA_PATH = "../data/chroma_reviews"
client = chromadb.PersistentClient(path=CHROMA_PATH)

# Drop and recreate to keep it reproducible
if "olist_reviews" in [c.name for c in client.list_collections()]:
    client.delete_collection("olist_reviews")

collection = client.create_collection(
    name="olist_reviews",
    metadata={"description": "Olist negative review comments (Portuguese)"},
)

# Add in batches (Chroma has a limit per add call)
BATCH = 500
for i in tqdm(range(0, len(reviews), BATCH), desc="Indexing"):
    end = min(i + BATCH, len(reviews))
    collection.add(
        ids=[str(x) for x in range(i, end)],
        documents=reviews["review_comment_message"].iloc[i:end].tolist(),
        embeddings=embeddings[i:end],
        metadatas=[
            {
                "order_id": str(row["order_id"]),
                "category": row["category"],
                "is_late": int(row["is_late"]),
            }
            for _, row in reviews.iloc[i:end].iterrows()
        ],
    )

print(f"Indexed {collection.count()} reviews")


Indexing:   0%|          | 0/18 [00:00<?, ?it/s]

Indexed 8760 reviews


## 5. Semantic search function

The function takes an English query, translates its meaning into embeddings, and returns the `n` most similar Portuguese reviews.

In [7]:
def search_reviews(query, n=5, category=None, is_late=None):
    """Semantic search over the review index.
    
    Args:
        query: Search query in English (or Portuguese)
        n: Number of results to return
        category: Optional filter by complaint category
        is_late: Optional filter by delivery status (0 or 1)
    """
    query_embedding = model.encode([query]).tolist()
    
    # Build metadata filter
    where = {}
    if category:
        where["category"] = category
    if is_late is not None:
        where["is_late"] = int(is_late)
    
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n,
        where=where if where else None,
    )
    
    # Format as a list of dicts
    hits = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        hits.append({
            "text": doc,
            "category": meta["category"],
            "is_late": meta["is_late"],
            "similarity": round(1 - dist, 3),
        })
    return hits


## 6. Example queries

These are the kinds of questions the old agent could NOT answer because they cut across categories.

In [8]:
# Query 1: packaging issues
print("=== Query: packaging problems ===")
for hit in search_reviews("packaging damaged box broken wrapping", n=3):
    print(f"[{hit['category']}, sim={hit['similarity']}]")
    print(f"  {hit['text'][:200]}")
    print()


=== Query: packaging problems ===
[broken_damaged, sim=-11.181]
  A embalagem chegou toda danificada. Tanto embrulho quanto a caixa do produto.

[broken_damaged, sim=-13.821]
  O produto veio embalado apenas por um frágil envelope, o que ocasionou o fato da caixa do cartucho ficar totalmente destruída.

[broken_damaged, sim=-13.821]
  O produto veio embalado apenas por um frágil envelope, o que ocasionou o fato da caixa do cartucho ficar totalmente destruída.



In [9]:
# Query 2: refund / money back
print("=== Query: refund / money back ===")
for hit in search_reviews("refund money back cancel return money", n=3):
    print(f"[{hit['category']}, sim={hit['similarity']}]")
    print(f"  {hit['text'][:200]}")
    print()


=== Query: refund / money back ===
[customer_service_issue, sim=-17.124]
  Recebi e-mail informando "Imprevisto durante a entrega do pedido" sendo que não tive nenhum contato apenas e-mail automáticos quero saber o que aconteceu com a entrega e quero meu dinheiro de volta

[customer_service_issue, sim=-17.793]
  Não recomendo!!! Cancelei a compra, fui cobrada e não obtive NENHUM retorno da loja. Quero devolver e receber o extorno. 

[customer_service_issue, sim=-18.311]
  Tentando cancelar o produto desde antes da emissão da nota fiscal. Não fui avisada que estava no correio, não tive nenhuma resposta do cancelamento e nao consigo devolver. 



In [10]:
# Query 3: seller communication — filter to customer_service_issue
print("=== Query: can't reach seller ===")
for hit in search_reviews(
    "cannot contact seller no response customer service",
    n=3,
    category="customer_service_issue",
):
    print(f"[{hit['category']}, sim={hit['similarity']}]")
    print(f"  {hit['text'][:200]}")
    print()


=== Query: can't reach seller ===
[customer_service_issue, sim=-10.452]
  Um verdadeiro descaso com o cliente. Mandei e-mail duas vezes para a loja e até agora não recebi nenhuma resposta. Nem telefone para contato é fornecido! Não recomendo nada que vem dessa loja.

[customer_service_issue, sim=-11.994]
  O pedido foi entregue parcial e o vendedor não entrou em contato comigo. Estou aguardando resposta da pág seguro

[customer_service_issue, sim=-12.218]
  Desisti da compra enquanto o produto ainda estava na transportadora e a loja nao atende meu pedido. Nao respondem minha solicitacao de código de envio para devolução. Nao retornam meus emails.



In [11]:
# Query 4: phantom delivery — reviews about system says delivered but not received
print("=== Query: system says delivered but not received ===")
for hit in search_reviews(
    "tracking says delivered but I never received",
    n=5,
    is_late=0,  # on-time orders only
):
    print(f"[{hit['category']}, sim={hit['similarity']}]")
    print(f"  {hit['text'][:200]}")
    print()


=== Query: system says delivered but not received ===
[never_arrived, sim=-16.509]
  por enquanto, não recebi minha encomenda apesar de estar indicado no rastreio que ja foi entregue.

[customer_service_issue, sim=-16.674]
  Não recebi o item que já contava como cancelado no momento que consultava meus pedidos.
Não obtive nenhuma resposta às mensagens enviadas

[customer_service_issue, sim=-16.703]
  Preciso efetuar a troca enviei msg e nada de resposta

[never_arrived, sim=-16.838]
  No acompanhamento de pedidos o status está como entregue, contudo o produto nunca chegou na minha casa. 

[never_arrived, sim=-16.857]
  Pode ter ocorrido falha no sistema mas, até a presente data, meu produto não foi entregue; embora conste o contrário no sistema de rastreamento.



## 7. Full agent with RAG — 4 tools

Now we rebuild the agent from notebook 04 but with an extra tool: `search_reviews`.
The agent can now answer theme-based questions the old 3-tool agent could not.

### The 4 tools
1. `get_category_counts` — counts per category, optionally filtered by delivery status
2. `get_sample_reviews` — random Portuguese reviews from a category
3. `search_reviews` — **NEW: semantic search by theme (not category)**
4. `translate_text` — Portuguese → English

### 7.1 Setup — Groq client (same as notebook 04)

In [12]:
from dotenv import load_dotenv
from openai import OpenAI
import json
import time

load_dotenv()

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "openai/gpt-oss-120b"

print("Groq key loaded:", "GROQ_API_KEY" in os.environ)


Groq key loaded: True


### 7.2 Tool functions

The first three are copied from notebook 04. `search_reviews` is defined above in Section 5.

In [13]:
def get_category_counts(filter_by="all"):
    """Count reviews per category. filter_by: 'late', 'on_time', or 'all'."""
    df = sample
    if filter_by == "late":
        df = df[df["is_late"] == 1]
    elif filter_by == "on_time":
        df = df[df["is_late"] == 0]
    return df["category"].value_counts().to_dict()


def get_sample_reviews(category, filter_by="all", n=3):
    """Return n random Portuguese review texts for a category."""
    df = sample
    if filter_by == "late":
        df = df[df["is_late"] == 1]
    elif filter_by == "on_time":
        df = df[df["is_late"] == 0]
    df = df[df["category"] == category]
    return df["review_comment_message"].sample(n=min(n, len(df))).tolist()


def translate_text(text):
    """Translate Portuguese to English via Groq."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a translator. Translate the Portuguese text to English. Return only the translation, no explanation."},
            {"role": "user", "content": text},
        ],
    )
    return response.choices[0].message.content


# search_reviews is already defined in Section 5 above


### 7.3 Tool schemas (JSON for the LLM)

Four tool definitions passed to the LLM.

In [14]:
CATEGORIES = [
    "never_arrived", "wrong_item", "misleading_desc", "late_delivery",
    "poor_quality", "broken_damaged", "customer_service_issue", "other",
]

category_counts_tool = {
    "type": "function",
    "function": {
        "name": "get_category_counts",
        "description": "Get the count of negative reviews in each of 8 complaint categories. Can filter by delivery status.",
        "parameters": {
            "type": "object",
            "properties": {
                "filter_by": {
                    "type": "string",
                    "enum": ["all", "late", "on_time"],
                    "description": "Filter by delivery status",
                },
            },
            "required": ["filter_by"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}

sample_reviews_tool = {
    "type": "function",
    "function": {
        "name": "get_sample_reviews",
        "description": "Get RANDOM sample review texts (in Portuguese) for a specific complaint category. Use this ONLY when the user asks for examples from a specific category, not for a specific theme. Always use translate_text before showing them to the user.",
        "parameters": {
            "type": "object",
            "properties": {
                "category": {
                    "type": "string",
                    "enum": CATEGORIES,
                    "description": "One of the 8 complaint categories",
                },
                "filter_by": {
                    "type": "string",
                    "enum": ["all", "late", "on_time"],
                },
                "n": {
                    "type": "integer",
                    "description": "Number of samples to return (2-5 recommended)",
                },
            },
            "required": ["category", "filter_by", "n"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}

search_reviews_tool = {
    "type": "function",
    "function": {
        "name": "search_reviews",
        "description": "Semantic search over all 8,760 reviews by MEANING. Use this when the user asks about a specific THEME that doesn't match one of the 8 categories (e.g. packaging, refund, communication, phantom delivery, specific complaints). This finds the MOST RELEVANT reviews, not random ones. Always translate results with translate_text before showing to user.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "English query describing the theme to search for",
                },
                "n": {
                    "type": "integer",
                    "description": "Number of reviews to return (2-5 recommended)",
                },
            },
            "required": ["query", "n"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}

translate_tool = {
    "type": "function",
    "function": {
        "name": "translate_text",
        "description": "Translate Portuguese review text to English.",
        "parameters": {
            "type": "object",
            "properties": {
                "text": {
                    "type": "string",
                    "description": "Portuguese text to translate",
                },
            },
            "required": ["text"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}

TOOLS = [category_counts_tool, sample_reviews_tool, search_reviews_tool, translate_tool]
FUNCTIONS = {
    "get_category_counts": get_category_counts,
    "get_sample_reviews": get_sample_reviews,
    "search_reviews": lambda query, n: search_reviews(query, n=n),
    "translate_text": translate_text,
}


### 7.4 Agent loop and system prompt

In [15]:
def run_agent(messages, max_steps=6):
    for step in range(1, max_steps + 1):
        # Retry on transient 503 / network errors
        response = None
        for attempt in range(3):
            try:
                response = client.chat.completions.create(
                    model=MODEL,
                    messages=messages,
                    tools=TOOLS,
                )
                break
            except Exception as error:
                if attempt == 2:
                    raise
                print(f"  retry {attempt + 1}: {type(error).__name__}")
                time.sleep(2)

        message = response.choices[0].message
        if not message.tool_calls:
            return message.content
        messages.append(message)
        for call in message.tool_calls:
            arguments = json.loads(call.function.arguments)
            try:
                result = FUNCTIONS[call.function.name](**arguments)
            except Exception as error:
                result = f"Error: {error}"
            print(f"  step {step}: {call.function.name}({arguments}) -> {str(result)[:150]}")
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result, default=str),
            })
    return "Stopped: too many steps."


ANALYST = """You are a data analyst for Olist, a Brazilian e-commerce marketplace.
You help managers understand why customers leave negative reviews.

You have access to 8,760 negative reviews with 8 complaint categories.
Reviews are in Portuguese. When showing examples, always translate them to English.

TOOL SELECTION:
- Use get_category_counts for questions about counts per category.
- Use get_sample_reviews ONLY when the user asks for random examples from a specific category.
- Use search_reviews when the user asks about a THEME (packaging, refund, communication, late delivery, phantom delivery, specific complaint type). This is almost always what you want for "show me examples about X".
- Use translate_text on EVERY Portuguese review before showing it to the user.

STRICT RULES:
1. Base every number on a tool result. Never guess numbers.
2. Base every recommendation on what the data shows (counts, examples).
3. If the data does not support a conclusion, say "The data does not show this."

Answer in 3-5 short sentences with:
- The key numbers
- 1-2 translated review examples when relevant
- A data-backed recommendation"""


### 7.5 Agent in action — questions that use the new RAG tool

These are the questions the old agent could NOT answer well:

In [16]:
question = "Show me 2 examples of customers complaining about packaging or damaged boxes. Translate them to English."
print(run_agent([
    {"role": "system", "content": ANALYST},
    {"role": "user", "content": question},
]))


  step 1: search_reviews({'n': 2, 'query': 'complaint about packaging damaged box'}) -> [{'text': 'A embalagem chegou toda danificada. Tanto embrulho quanto a caixa do produto.', 'category': 'broken_damaged', 'is_late': 0, 'similarity': -
  step 2: translate_text({'text': 'A embalagem chegou toda danificada. Tanto embrulho quanto a caixa do produto.'}) -> The packaging arrived completely damaged. Both the wrapping and the product's box.
  step 3: translate_text({'text': 'A embalagem veio danificada assim como o produto, quebrado. Entrei em contato com a loja e ainda não tive nenhuma resposta.'}) -> The packaging arrived damaged as well as the product, broken. I contacted the store and have not received any response yet.
**Key numbers**: Packaging‑related complaints fall within the “broken/damaged” category, which accounts for ≈ 12% of all negative reviews (about 1,050 reviews).  

**Examples**  
1. “The packaging arrived completely damaged. Both the wrapping and the product’s box.”  
2

In [17]:
question = "Find reviews where customers say the tracking shows delivered but they never received the item. Give 2 translated examples."
print(run_agent([
    {"role": "system", "content": ANALYST},
    {"role": "user", "content": question},
]))


  step 1: search_reviews({'n': 2, 'query': 'tracking shows delivered but never received'}) -> [{'text': 'Pode ter ocorrido falha no sistema mas, até a presente data, meu produto não foi entregue; embora conste o contrário no sistema de rastream
  step 2: translate_text({'text': 'Pode ter ocorrido falha no sistema mas, até a presente data, meu produto não foi entregue; embora conste o contrário no sistema de rastreamento.'}) -> There may have been a system failure, but as of today, my product has not been delivered; although the tracking system says otherwise.
  step 3: translate_text({'text': 'O produto ainda não chegou, e não tenho nenhuma informação a respeito do mesmo. Total descaso!\r\nNa rastreabilidade aparece apenas "dificuldades na entrega", sem dar nenhum prazo...'}) -> The product hasn't arrived yet, and I have no information about it. Total neglect! The tracking only shows “delivery difficulties,” without giving any
- Example 1: “There may have been a system failure, but as 

In [18]:
question = "What are customers saying about refunds or getting their money back? Give 2 translated examples."
print(run_agent([
    {"role": "system", "content": ANALYST},
    {"role": "user", "content": question},
]))


  step 1: search_reviews({'n': 5, 'query': 'refund not received money back'}) -> [{'text': 'Recebi e-mail informando "Imprevisto durante a entrega do pedido" sendo que não tive nenhum contato apenas e-mail automáticos quero saber o
  step 2: translate_text({'text': 'Recebi e-mail informando "Imprevisto durante a entrega do pedido" sendo que não tive nenhum contato apenas e-mail automáticos quero saber o que aconteceu com a entrega e quero meu dinheiro de volta'}) -> I received an email stating "Unexpected issue during the order delivery," even though I had no contact, only automatic emails. I want to know what hap
  step 3: translate_text({'text': 'Não recebi, pedi reembolso e o crédito não entrou na conta'}) -> I didn't receive it, I requested a refund, and the credit hasn't been posted to the account.
- The data does not provide a specific count for refund‑related complaints, but several negative reviews mention refunds.  
- Example 1: “I received an email stating *‘Unexpected issue 

## 8. Summary — why RAG matters

### Before RAG (notebook 04)
The agent could only group reviews by one of 8 pre-defined categories. When a user asked "what do customers say about packaging?", the agent had to guess which category might be relevant (`broken_damaged`?) and pull random reviews — many of which weren't about packaging at all.

### With RAG (this notebook)
The agent searches **semantically**: it finds the reviews most similar in meaning to the user's question, regardless of category. "Packaging complaints" now returns reviews that actually mention packaging, from whichever categories they happen to fall in.

### What this unlocks
- **Theme-based questions** the old categories don't capture (packaging, refunds, communication)
- **Cross-category search** (phantom delivery spans `never_arrived` and `late_delivery`)
- **More trustworthy examples** (relevant, not random)
- **Better demo stories** — the agent looks like it actually read the reviews

### Limitations
- Multilingual embeddings are good but not perfect — some Portuguese nuance may be missed
- The embedding model handles short reviews best; very long ones get compressed
- Chroma index needs to be rebuilt if reviews change (handled by the if-cached check)

### Next step
Expose this through the Streamlit app so recruiters can try their own queries live.